# 02 — Data Preprocessing
**Preparing raw data for machine learning**

Steps: drop redundant columns → encode categorical features → separate X and y → train-test split


## 1. Load data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/delhi_ncr_aqi_dataset.csv")
print(f"Loaded: {df.shape}")

Loaded: (201664, 25)


## 2. Drop redundant columns
Dropped `datetime` and `date` — time info already extracted into year/month/day/hour columns.
Dropped `station`, `latitude`, `longitude` — city column already captures location.


In [2]:
df = df.drop(columns=['datetime', 'date', 'station', 'latitude', 'longitude'])
print(f"After drop: {df.shape}")
print(list(df.columns))

After drop: (201664, 20)
['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'season', 'city', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'visibility', 'aqi', 'aqi_category']


## 3. Encode categorical features
Three text columns need to become numbers before the model can use them.

- `day_of_week` — ordinal encoding (Mon=0 → Sun=6), order is meaningful
- `season` — ordinal encoding ordered by pollution severity (monsoon=0 → winter=3)
- `city` — label encoding, no meaningful order between cities
- `aqi_category` — ordinal encoding for use as classification target only


In [3]:
# day_of_week — ordered Mon to Sun
day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
           'Friday':4,'Saturday':5,'Sunday':6}
df['day_of_week'] = df['day_of_week'].map(day_map)

# season — ordered by pollution severity
season_map = {'monsoon':0,'summer':1,'post_monsoon':2,'winter':3}
df['season'] = df['season'].map(season_map)

# city — alphabetical label encoding
df['city'] = LabelEncoder().fit_transform(df['city'])

# aqi_category — classification target only, not used as input feature
cat_map = {'Good':0,'Satisfactory':1,'Moderate':2,'Poor':3,'Very Poor':4,'Severe':5}
df['aqi_category_encoded'] = df['aqi_category'].map(cat_map)

print("Encoding done. dtypes:")
print(df.dtypes)

Encoding done. dtypes:
year                      int64
month                     int64
day                       int64
hour                      int64
day_of_week               int64
is_weekend                int64
season                    int64
city                      int64
pm25                    float64
pm10                    float64
no2                     float64
so2                     float64
co                      float64
o3                      float64
temperature             float64
humidity                  int64
wind_speed              float64
visibility              float64
aqi                       int64
aqi_category             object
aqi_category_encoded      int64
dtype: object


## 4. Separate features (X) and targets (y)
Two separate prediction tasks:
- **Regression** → predict exact AQI number
- **Classification** → predict AQI category (0–5)

`aqi_category` is dropped from X — it is derived directly from `aqi` and would cause data leakage.


In [4]:
X = df.drop(columns=['aqi', 'aqi_category', 'aqi_category_encoded'])
y_reg = df['aqi']
y_clf = df['aqi_category_encoded']

print(f"X shape:     {X.shape}  — 18 input features")
print(f"y_reg shape: {y_reg.shape}  — regression target")
print(f"y_clf shape: {y_clf.shape}  — classification target")

X shape:     (201664, 18)  — 18 input features
y_reg shape: (201664,)  — regression target
y_clf shape: (201664,)  — classification target


## 5. Train-test split
80% training, 20% test. `random_state=42` ensures reproducibility.
The test set is held out completely until model evaluation.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")

Training set: 161,331 rows
Test set:     40,333 rows


## 6. Weather-only feature set
During modeling, I discovered that including pollutant columns gives near-perfect scores
because AQI is mathematically derived from those pollutants — this is data leakage.

I created a second feature set using only weather and time features for a genuinely
challenging prediction task: predict AQI without sensor readings.


In [6]:
weather_features = ['year','month','day','hour',
                    'day_of_week','is_weekend','season','city',
                    'temperature','humidity','wind_speed','visibility']

X_weather = df[weather_features]
X_tr, X_te, y_tr, y_te = train_test_split(X_weather, y_reg, test_size=0.2, random_state=42)

print(f"Weather-only training set: {X_tr.shape}")
print(f"Weather-only test set:     {X_te.shape}")

Weather-only training set: (161331, 12)
Weather-only test set:     (40333, 12)
